# Train the ISL sign-recognition model

This notebook teaches a model to recognise Indian Sign Language words **for anyone**, using the free
**INCLUDE** dataset (videos of 263 ISL words signed by Deaf students of St. Louis School for the Deaf, Chennai —
IIT Madras / AI4Bharat, CC BY 4.0). You don't need to know sign language: the dataset is the teacher.

**What to do**
1. `Runtime → Change runtime type → T4 GPU` (free) → Save.
2. Run each cell from top to bottom (▶ button, or `Runtime → Run all`).
3. At the end a file **`isl_model.zip`** downloads. Unzip it into your app folder so you get
   `ISL_Avatar_App/model/isl_model.onnx` and `isl_model.json`, then restart the app.

**Time:** quick start ≈ 30–45 min, everything ≈ 1.5–2.5 h (mostly downloading and reading the videos).
Progress is saved to your Google Drive, so if Colab disconnects just run all cells again — it continues where it stopped.

In [ ]:
#@title 1 · Setup (≈1 min) { display-mode: "form" }
SAVE_TO_GOOGLE_DRIVE = True  #@param {type:"boolean"}
LOCAL_TEST_ZIPS = ""         # leave empty (used only for offline testing of this notebook)

import os, sys, shutil, subprocess, time, glob
IN_COLAB = 'google.colab' in sys.modules
if SAVE_TO_GOOGLE_DRIVE and IN_COLAB and not LOCAL_TEST_ZIPS:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/ISL_training'
else:
    WORK = os.path.abspath('ISL_training')
TMP = '/content/tmp' if IN_COLAB else os.path.abspath('tmp_isl')
for d in (WORK, f'{WORK}/landmarks', f'{WORK}/done', TMP, 'models'): os.makedirs(d, exist_ok=True)

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode: print(r.stdout[-2000:], r.stderr[-2000:]); raise RuntimeError('command failed: ' + cmd)
    return r.stdout

try:
    import mediapipe, onnx, onnxruntime
except ImportError:
    sh(f'{sys.executable} -m pip -q install "mediapipe>=0.10.14,<0.11" onnx onnxruntime')
for url in ['https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task',
            'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task']:
    dst = 'models/' + url.rsplit('/', 1)[1]
    if not os.path.exists(dst): sh(f'wget -q -O {dst} {url}')
import torch
print('Work folder:', WORK)
gpu = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True, capture_output=True, text=True).stdout.strip()
print('GPU:', gpu or 'none (training will be much slower — Runtime → Change runtime type → T4 GPU)')
print('CPUs:', os.cpu_count())

In [ ]:
%%writefile islfeat.py
"""Python port of signmath.js (tracker -> features -> resampled sequence -> model vector).
Must stay numerically identical to the JavaScript used in the app; tested by the parity check."""
import math
import numpy as np

N = 24
RAISED = 1.5
DROP_BELOW = 2.0
OUTSIDE_X, OUTSIDE_UP = 3.2, 3.5
FEAT_VERSION, FEAT_PER_HAND, FEAT_DIM = 1, 82, 164
CHAINS = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16], [17, 18, 19, 20]]


def _hypot(*a):
    return math.sqrt(sum(x * x for x in a))


def hand_angles(shape):
    def P(j):
        return (0.0, 0.0, 0.0) if j == 0 else (shape[(j-1)*3], shape[(j-1)*3+1], shape[(j-1)*3+2] * 0.5)
    def sub(a, b):
        return (a[0]-b[0], a[1]-b[1], a[2]-b[2])
    def ang(u, v):
        d = u[0]*v[0] + u[1]*v[1] + u[2]*v[2]
        n = _hypot(*u) * _hypot(*v) or 1.0
        return math.acos(max(-1.0, min(1.0, d / n)))
    out = []
    for c in CHAINS:
        pts = [P(j) for j in [0] + c]
        for i in range(1, 4):
            out.append(ang(sub(pts[i], pts[i-1]), sub(pts[i+1], pts[i])))
    for i in range(4):
        a, b = CHAINS[i], CHAINS[i+1]
        out.append(ang(sub(P(a[3]), P(a[0])), sub(P(b[3]), P(b[0]))))
    return out


class Tracker:
    """hands: list of dicts {label, score, lm: [(x,y,z)*21]} in normalised image coords; pose: dict {11:(x,y,vis), 12:(x,y,vis)} or None."""
    def __init__(self):
        self.body = dict(cx=0.0, cy=0.0, s=0.0, ok=False, seen=False)
        self.prev = dict(R=None, L=None, tR=-1e9, tL=-1e9)

    def _update_body(self, pose, W, H):
        b = self.body
        a = pose.get(11) if pose else None
        c = pose.get(12) if pose else None
        if a and c and a[2] > 0.5 and c[2] > 0.5:
            cx, cy = (a[0] + c[0]) / 2 * W, (a[1] + c[1]) / 2 * H
            s = _hypot((a[0] - c[0]) * W, (a[1] - c[1]) * H)
            if s > 0.08 * W and s < 0.9 * W and (not b['seen'] or abs(s / b['s'] - 1) < 0.6 or not b['ok']):
                k = 0.35 if b['seen'] else 1.0
                b['cx'] += (cx - b['cx']) * k; b['cy'] += (cy - b['cy']) * k; b['s'] += (s - b['s']) * k
                b['ok'] = True; b['seen'] = True
                return
        b['ok'] = False
        if not b['seen']:
            b['cx'], b['cy'], b['s'] = W / 2, H * 0.62, W * 0.3

    def _hand_feat(self, lm, W, H):
        b = self.body
        w = lm[0]
        size = _hypot((lm[9][0] - w[0]) * W, (lm[9][1] - w[1]) * H) or 1.0
        shape = []
        for j in range(1, 21):
            shape += [(lm[j][0] - w[0]) * W / size, (lm[j][1] - w[1]) * H / size, (lm[j][2] - w[2]) * W / size]
        return dict(loc=[(w[0] * W - b['cx']) / b['s'], (w[1] * H - b['cy']) / b['s']], shape=shape, ang=hand_angles(shape))

    def frame(self, hands, pose, W, H, t):
        self._update_body(pose, W, H)
        f = dict(t=t, R=None, L=None)
        c = [dict(h=self._hand_feat(h['lm'], W, H), label=h.get('label') or '', score=h.get('score', 0.5)) for h in (hands or [])]
        c = [x for x in c if abs(x['h']['loc'][0]) < OUTSIDE_X and x['h']['loc'][1] > -OUTSIDE_UP and x['h']['loc'][1] < DROP_BELOW]
        if len(c) == 2 and _hypot(c[0]['h']['loc'][0] - c[1]['h']['loc'][0], c[0]['h']['loc'][1] - c[1]['h']['loc'][1]) < 0.12:
            c = [c[0] if c[0]['score'] >= c[1]['score'] else c[1]]
        def label_side(x):
            if x['label'] == 'Left': return 'R'
            if x['label'] == 'Right': return 'L'
            return 'R' if x['h']['loc'][0] < 0 else 'L'
        p = self.prev
        pR = p['R'] if t - p['tR'] < 400 else None
        pL = p['L'] if t - p['tL'] < 400 else None
        def d(h, q):
            return None if q is None else _hypot(h['loc'][0] - q['loc'][0], h['loc'][1] - q['loc'][1])
        if len(c) == 2:
            a, b = c
            if pR is not None and pL is not None:
                if d(a['h'], pR) + d(b['h'], pL) > d(a['h'], pL) + d(b['h'], pR): a, b = b, a
            elif label_side(a) == label_side(b):
                if a['h']['loc'][0] > b['h']['loc'][0]: a, b = b, a
            elif label_side(a) == 'L':
                a, b = b, a
            f['R'], f['L'] = a['h'], b['h']
        elif len(c) == 1:
            x = c[0]; dR, dL = d(x['h'], pR), d(x['h'], pL)
            if dR is not None and dL is not None: side = 'R' if dR <= dL else 'L'
            elif dR is not None and dR < 0.8: side = 'R'
            elif dL is not None and dL < 0.8: side = 'L'
            else: side = label_side(x)
            f[side] = x['h']
        if f['R'] is not None: p['R'], p['tR'] = f['R'], t
        if f['L'] is not None: p['L'], p['tL'] = f['L'], t
        return f


def has_hand(f): return f['R'] is not None or f['L'] is not None
def is_raised(f): return any(f[k] is not None and f[k]['loc'][1] < RAISED for k in ('R', 'L'))


def _lerp_hand(a, b, k):
    if a is None or b is None:
        return (a if a is not None else b) if k < 0.5 else (b if b is not None else a)
    return dict(loc=[a['loc'][i] + (b['loc'][i] - a['loc'][i]) * k for i in range(2)],
                shape=[v + (b['shape'][i] - v) * k for i, v in enumerate(a['shape'])],
                ang=[v + (b['ang'][i] - v) * k for i, v in enumerate(a['ang'])])


def to_sample(frames, n=N):
    def first_last(pred):
        idx = [i for i, f in enumerate(frames) if pred(f)]
        return (idx[0], idx[-1]) if idx else None
    span = first_last(is_raised) or first_last(has_hand)
    if not span: return None
    seg = frames[span[0]:span[1] + 1]
    if sum(1 for f in seg if has_hand(f)) < 4: return None
    t0, t1 = seg[0]['t'], seg[-1]['t']
    out = []
    for i in range(n):
        tt = t0 + (t1 - t0) * i / (n - 1)
        j = 0
        while j < len(seg) - 2 and seg[j + 1]['t'] < tt: j += 1
        a, b = seg[j], seg[min(j + 1, len(seg) - 1)]
        k = min(1.0, max(0.0, (tt - a['t']) / (b['t'] - a['t']))) if b['t'] > a['t'] else 0.0
        out.append(dict(R=_lerp_hand(a['R'], b['R'], k), L=_lerp_hand(a['L'], b['L'], k)))
    return out


def _mirror(h):
    return None if h is None else dict(loc=[-h['loc'][0], h['loc'][1]], shape=[-v if i % 3 == 0 else v for i, v in enumerate(h['shape'])], ang=h['ang'])


def to_vector(seq):
    out = np.zeros((len(seq), FEAT_DIM), dtype=np.float32)
    for t, f in enumerate(seq):
        for side, k in enumerate(('R', 'L')):
            h = f[k]
            if h is None: continue
            o = side * FEAT_PER_HAND
            out[t, o] = 1; out[t, o+1:o+3] = h['loc']; out[t, o+3:o+63] = h['shape']; out[t, o+63:o+82] = h['ang']
    return out


def features_from_raw(raw_frames, n):
    """raw_frames: list of {t, W, H, hands:[{label,score,lm}], pose:{11:(x,y,v),12:(x,y,v)}|None} -> [n x 164] or None"""
    tr = Tracker()
    frames = [tr.frame(r['hands'], r['pose'], r['W'], r['H'], r['t']) for r in raw_frames]
    seq = to_sample(frames, n)
    return None if seq is None else to_vector(seq)

In [ ]:
%%writefile isltrain.py
"""ISL sign model training: videos -> MediaPipe landmarks -> augmented feature sequences -> 1D-CNN -> ONNX.
Used by the Colab notebook (train_isl_model.ipynb). Feature maths lives in islfeat.py (identical to the app's signmath.js)."""
import gzip, json, math, os, re, random, time, zipfile
from collections import Counter, defaultdict
import numpy as np
import islfeat as F

VIDEO_EXT = ('.mov', '.mp4', '.avi', '.mkv', '.m4v', '.webm')
T = 32                       # frames per sign given to the model (app resamples to the same length)

# ----------------------------------------------------------------------------- landmark extraction
_HL = _PL = None
_TS = 0


def _init_worker(hand_model, pose_model):
    """Create one hand + pose landmarker per worker process (VIDEO mode)."""
    global _HL, _PL
    from mediapipe.tasks import python as mpt
    from mediapipe.tasks.python import vision
    _HL = vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(
        base_options=mpt.BaseOptions(model_asset_path=hand_model), running_mode=vision.RunningMode.VIDEO,
        num_hands=2, min_hand_detection_confidence=0.5, min_hand_presence_confidence=0.5, min_tracking_confidence=0.5))
    _PL = vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(
        base_options=mpt.BaseOptions(model_asset_path=pose_model), running_mode=vision.RunningMode.VIDEO, num_poses=1))


def extract_video(path, max_width=640, target_fps=15):
    """Run hand + pose tracking on one video. Returns list of raw frames (same format the app produces)."""
    global _TS
    import cv2, mediapipe as mp
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    step = max(1, round(fps / target_fps))
    frames, i = [], 0
    r4 = lambda v: round(float(v), 4)
    while True:
        ok, img = cap.read()
        if not ok: break
        if i % step == 0:
            h, w = img.shape[:2]
            if w > max_width:
                img = cv2.resize(img, (max_width, round(h * max_width / w)), interpolation=cv2.INTER_AREA); h, w = img.shape[:2]
            rgb = np.ascontiguousarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            _TS += 40                                     # MediaPipe needs strictly increasing timestamps (ms)
            im = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            hr, pr = _HL.detect_for_video(im, _TS), _PL.detect_for_video(im, _TS)
            hands = [dict(label=hr.handedness[k][0].category_name, score=r4(hr.handedness[k][0].score),
                          lm=[[r4(p.x), r4(p.y), r4(p.z)] for p in lm]) for k, lm in enumerate(hr.hand_landmarks)]
            pose = None
            if pr.pose_landmarks:
                pl = pr.pose_landmarks[0]
                pose = {11: [r4(pl[11].x), r4(pl[11].y), r4(pl[11].visibility or 1)], 12: [r4(pl[12].x), r4(pl[12].y), r4(pl[12].visibility or 1)]}
            frames.append(dict(t=round(i * 1000 / fps, 1), W=w, H=h, hands=hands, pose=pose))
        i += 1
    cap.release()
    return frames


def _extract_job(args):
    path, out_path, meta = args
    try:
        frames = extract_video(path)
        with gzip.open(out_path, 'wt') as f: json.dump(dict(meta, frames=frames), f)
        return out_path, len(frames), None
    except Exception as e:  # keep going; report at the end
        return out_path, 0, repr(e)


def label_from_path(path):
    """INCLUDE layout: <Category>/<NN. Word>/<video>. Label = folder name without the number prefix."""
    parent = os.path.basename(os.path.dirname(path))
    cat = os.path.basename(os.path.dirname(os.path.dirname(path)))
    label = re.sub(r'^\s*\d+\s*[.)_-]?\s*', '', parent).strip().lower()
    return label or parent.lower(), cat


def extract_folder(video_root, out_dir, hand_model, pose_model, workers=None, log=print):
    """Extract landmarks for every video under video_root into out_dir (one .json.gz per video). Skips finished ones."""
    from multiprocessing import Pool
    os.makedirs(out_dir, exist_ok=True)
    jobs = []
    for root, _, files in os.walk(video_root):
        for fn in sorted(files):
            if fn.lower().endswith(VIDEO_EXT) and not fn.startswith('._'):
                p = os.path.join(root, fn)
                label, cat = label_from_path(p)
                key = re.sub(r'[^A-Za-z0-9]+', '_', os.path.relpath(p, video_root))
                out = os.path.join(out_dir, key + '.json.gz')
                if not os.path.exists(out): jobs.append((p, out, dict(label=label, category=cat, video=os.path.relpath(p, video_root))))
    if not jobs: return 0
    workers = workers or max(1, os.cpu_count() or 1)
    t0, done, errors = time.time(), 0, []
    with Pool(workers, initializer=_init_worker, initargs=(hand_model, pose_model)) as pool:
        for out, n, err in pool.imap_unordered(_extract_job, jobs):
            done += 1
            if err: errors.append((out, err))
            if done % 25 == 0 or done == len(jobs):
                el = time.time() - t0
                log(f'  {done}/{len(jobs)} videos  ({el/done:.1f}s each, ~{el/done*(len(jobs)-done)/60:.0f} min left)')
    for out, err in errors[:5]: log('  failed:', out, err)
    return len(jobs) - len(errors)


def load_landmarks(lm_dir):
    items = []
    for fn in sorted(os.listdir(lm_dir)):
        if fn.endswith('.json.gz'):
            with gzip.open(os.path.join(lm_dir, fn), 'rt') as f: d = json.load(f)
            for fr in d['frames']:
                if fr['pose']: fr['pose'] = {int(k): v for k, v in fr['pose'].items()}
            items.append(d)
    return items


# ----------------------------------------------------------------------------- augmentation
def augment(frames, rng, mirror=None):
    """Simulate a different camera/person/take: mirror (left-handed), rotate, scale, shift, jitter, speed changes, dropped frames."""
    if mirror is None: mirror = rng.random() < 0.5
    rot, sc = rng.normal(0, 0.07), rng.uniform(0.8, 1.2)
    tx, ty = rng.normal(0, 0.05), rng.normal(0, 0.05)
    jit, fdrop, hdrop = rng.uniform(0.0, 0.004), rng.uniform(0, 0.12), rng.uniform(0, 0.06)
    c, s = math.cos(rot), math.sin(rot)
    n = len(frames)
    # random temporal crop + non-uniform speed (time warp)
    a = int(rng.integers(0, max(1, n // 10) + 1)); b = n - int(rng.integers(0, max(1, n // 10) + 1))
    sel = frames[a:max(b, a + 2)]
    speeds = rng.uniform(0.7, 1.4, size=4)
    out, t = [], 0.0
    for k, fr in enumerate(sel):
        if k and rng.random() < fdrop: continue
        W, H = fr['W'], fr['H']
        def tf(x, y, z=0.0):
            px, py = x * W - W / 2, y * H - H / 2
            if mirror: px = -px
            qx, qy = (c * px - s * py) * sc + tx * W, (s * px + c * py) * sc + ty * H
            return [(qx + W / 2) / W + rng.normal(0, jit), (qy + H / 2) / H + rng.normal(0, jit), z * sc + rng.normal(0, jit)]
        hands = []
        for h in fr['hands']:
            if rng.random() < hdrop: continue
            lab = h['label']
            if mirror: lab = {'Left': 'Right', 'Right': 'Left'}.get(lab, lab)
            hands.append(dict(label=lab, score=h['score'], lm=[tf(*p) for p in h['lm']]))
        pose = None
        if fr['pose']:
            p11, p12 = fr['pose'][11], fr['pose'][12]
            if mirror: p11, p12 = p12, p11
            q11, q12 = tf(p11[0], p11[1]), tf(p12[0], p12[1])
            pose = {11: [q11[0], q11[1], p11[2]], 12: [q12[0], q12[1], p12[2]]}
        if k: t += (sel[k]['t'] - sel[k - 1]['t']) * speeds[min(3, 4 * k // len(sel))]
        out.append(dict(t=t, W=W, H=H, hands=hands, pose=pose))
    return out


def _feat_job(args):
    frames, n_aug, seed, mirror_test = args
    rng = np.random.default_rng(seed)
    vecs = []
    v = F.features_from_raw(frames, T)
    if v is not None: vecs.append(v)
    if mirror_test:   # clean mirrored copy, used to measure left-handed performance
        v = F.features_from_raw(augment_clean_mirror(frames), T)
        return vecs, ([v] if v is not None else [])
    for _ in range(n_aug):
        v = F.features_from_raw(augment(frames, rng), T)
        if v is not None: vecs.append(v)
    return vecs, []


def augment_clean_mirror(frames):
    out = []
    for fr in frames:
        hands = [dict(label={'Left': 'Right', 'Right': 'Left'}.get(h['label'], h['label']), score=h['score'], lm=[[1 - x, y, z] for x, y, z in h['lm']]) for h in fr['hands']]
        pose = None
        if fr['pose']:
            p11, p12 = fr['pose'][11], fr['pose'][12]
            pose = {11: [1 - p12[0], p12[1], p12[2]], 12: [1 - p11[0], p11[1], p11[2]]}
        out.append(dict(t=fr['t'], W=fr['W'], H=fr['H'], hands=hands, pose=pose))
    return out


def split_items(items, test_frac=0.2, seed=0, official=None):
    """Stratified split by video. If official=(train_names, test_names) is given, use those file names instead."""
    if official:
        tr_names, te_names = official
        tr = [i for i, d in enumerate(items) if os.path.basename(d['video']).lower() in tr_names]
        te = [i for i, d in enumerate(items) if os.path.basename(d['video']).lower() in te_names]
        if len(tr) > 0.5 * len(items) and te: return tr, te
    rng = random.Random(seed); by = defaultdict(list)
    for i, d in enumerate(items): by[d['label']].append(i)
    tr, te = [], []
    for lab, idx in sorted(by.items()):
        rng.shuffle(idx); k = max(1, round(len(idx) * test_frac)) if len(idx) >= 3 else 0
        te += idx[:k]; tr += idx[k:]
    return tr, te


def build_arrays(items, labels, idx, n_aug, seed, workers=None, mirror_test=False):
    from multiprocessing import Pool
    lab2i = {l: i for i, l in enumerate(labels)}
    jobs = [(items[i]['frames'], n_aug, seed * 100003 + i, mirror_test) for i in idx]
    X, Y, XM, YM = [], [], [], []
    with Pool(workers or max(1, os.cpu_count() or 1)) as pool:
        for i, (vecs, mvecs) in zip(idx, pool.imap(_feat_job, jobs, chunksize=4)):
            y = lab2i[items[i]['label']]
            X += vecs; Y += [y] * len(vecs); XM += mvecs; YM += [y] * len(mvecs)
    cat = lambda a: np.stack(a).astype(np.float16) if a else np.zeros((0, T, F.FEAT_DIM), np.float16)
    return cat(X), np.array(Y, np.int64), cat(XM), np.array(YM, np.int64)


# ----------------------------------------------------------------------------- model
def make_model(n_classes, d=192, blocks=4, drop=0.25):
    import torch, torch.nn as nn

    class Block(nn.Module):
        def __init__(s, d, k, drop):
            super().__init__()
            s.dw = nn.Conv1d(d, d, k, padding=k // 2, groups=d); s.bn = nn.BatchNorm1d(d)
            s.pw = nn.Conv1d(d, 2 * d, 1); s.pw2 = nn.Conv1d(2 * d, d, 1); s.act = nn.GELU(); s.drop = nn.Dropout(drop)
        def forward(s, x):
            return x + s.drop(s.pw2(s.act(s.pw(s.bn(s.dw(x))))))

    class SignNet(nn.Module):
        """Per-frame embedding -> temporal depthwise-separable convolutions -> mean+max pooling -> classes."""
        def __init__(s):
            super().__init__()
            s.norm = nn.LayerNorm(F.FEAT_DIM)
            s.inp = nn.Sequential(nn.Linear(F.FEAT_DIM, d), nn.GELU(), nn.Dropout(drop))
            s.blocks = nn.Sequential(*[Block(d, 5 if i % 2 == 0 else 3, drop) for i in range(blocks)])
            s.head = nn.Sequential(nn.Dropout(drop), nn.Linear(2 * d, n_classes))
        def forward(s, x):                        # x: [B, T, 164]
            h = s.inp(s.norm(x)).transpose(1, 2)  # [B, d, T]
            h = s.blocks(h)
            return s.head(torch.cat([h.mean(2), h.amax(2)], 1))
    return SignNet()


def train(Xtr, Ytr, Xte, Yte, n_classes, epochs=60, batch=128, lr=2e-3, log=print, seed=0):
    import torch, torch.nn as nn
    torch.manual_seed(seed)
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = make_model(n_classes).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    steps = epochs * math.ceil(len(Xtr) / batch)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.1)
    lossf = nn.CrossEntropyLoss(label_smoothing=0.1)
    Xtr_t, Ytr_t = torch.tensor(Xtr), torch.tensor(Ytr)
    best, best_state = -1, None
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xtr_t)); tot = 0.0
        for k in range(0, len(perm), batch):
            b = perm[k:k + batch]
            x, y = Xtr_t[b].float().to(dev), Ytr_t[b].to(dev)
            loss = lossf(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step(); tot += loss.item() * len(b)
        if (ep + 1) % 5 == 0 or ep == epochs - 1:
            acc = evaluate(model, Xte, Yte)['top1'] if len(Xte) else float('nan')
            log(f'  epoch {ep+1:3d}/{epochs}  loss {tot/len(perm):.3f}  test accuracy {acc*100:.1f}%')
    return model


def predict(model, X, batch=512):
    import torch
    dev = next(model.parameters()).device; model.eval(); out = []
    with torch.no_grad():
        for k in range(0, len(X), batch): out.append(torch.softmax(model(torch.tensor(X[k:k + batch]).float().to(dev)), 1).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, 0))


def evaluate(model, X, Y, labels=None):
    P = predict(model, X)
    top = np.argsort(-P, 1)
    r = dict(top1=float((top[:, 0] == Y).mean()), top3=float((top[:, :3] == Y[:, None]).any(1).mean()), n=len(Y))
    if labels is not None:
        conf = Counter((labels[y], labels[p]) for y, p in zip(Y, top[:, 0]) if y != p)
        r['confusions'] = [f'{a} -> {b} ({c})' for (a, b), c in conf.most_common(12)]
        per = defaultdict(list)
        for y, p in zip(Y, top[:, 0]): per[labels[y]].append(p == y)
        r['worst'] = sorted(((float(np.mean(v)), k) for k, v in per.items()))[:10]
        # confidence: how often the top guess is right when the model is >= 50% sure
        conf_ok = P.max(1) >= 0.5
        r['confident_share'] = float(conf_ok.mean())
        r['confident_accuracy'] = float((top[conf_ok, 0] == Y[conf_ok]).mean()) if conf_ok.any() else float('nan')
    return r


def export_onnx(model, path, labels, extra_meta):
    """Export to ONNX for the browser, then check onnxruntime gives the same outputs as PyTorch."""
    import torch, onnxruntime as ort
    model = model.cpu().eval()
    x = torch.randn(1, T, F.FEAT_DIM)
    kw = dict(input_names=['x'], output_names=['logits'], opset_version=17, dynamic_axes={'x': {0: 'batch'}, 'logits': {0: 'batch'}})
    try:
        torch.onnx.export(model, (x,), path, dynamo=False, **kw)
    except TypeError:
        torch.onnx.export(model, (x,), path, **kw)
    sess = ort.InferenceSession(path, providers=['CPUExecutionProvider'])
    xb = torch.randn(4, T, F.FEAT_DIM)
    diff = float(np.abs(sess.run(None, {'x': xb.numpy()})[0] - model(xb).detach().numpy()).max())
    meta = dict(format='isl-model-v1', feat_version=F.FEAT_VERSION, T=T, feat_dim=F.FEAT_DIM, labels=labels, onnx_check_max_diff=diff, **extra_meta)
    with open(os.path.join(os.path.dirname(path), 'isl_model.json'), 'w') as f: json.dump(meta, f, indent=1)
    return diff

In [ ]:
#@title 2 · Choose which signs to learn { display-mode: "form" }
DATASET = "Quick start (5 categories, ~19 GB download)"  #@param ["Quick start (5 categories, ~19 GB download)", "Everything (15 categories, all 263 signs, ~57 GB download)"]
ZIPS = {"Adjectives": 8, "Animals": 2, "Clothes": 2, "Colours": 2, "Days_and_Time": 3, "Electronics": 2, "Greetings": 2, "Home": 4, "Jobs": 2, "Means_of_Transportation": 2, "People": 5, "Places": 4, "Pronouns": 2, "Seasons": 1, "Society": 3}
QUICK = ["Greetings", "Pronouns", "Days_and_Time", "Society", "People"]
cats = QUICK if DATASET.startswith('Quick') else list(ZIPS)
zip_names = [f'{c}_{i}of{ZIPS[c]}.zip' for c in cats for i in range(1, ZIPS[c] + 1)]
if LOCAL_TEST_ZIPS:
    zip_names = sorted(os.path.basename(p) for p in glob.glob(os.path.join(LOCAL_TEST_ZIPS, '*.zip')))
print(len(zip_names), 'files to process:', ', '.join(zip_names))

In [ ]:
#@title 3 · Download videos and read hand movements (the long step) { display-mode: "form" }
import sys; sys.path.insert(0, '.')
import importlib, isltrain as L; importlib.reload(L)
HAND, POSE = os.path.abspath('models/hand_landmarker.task'), os.path.abspath('models/pose_landmarker_lite.task')
t_all = time.time()
for k, name in enumerate(zip_names, 1):
    marker = f'{WORK}/done/{name}.done'
    if os.path.exists(marker):
        print(f'[{k}/{len(zip_names)}] {name}: already done'); continue
    zp, vdir = f'{TMP}/{name}', f'{TMP}/videos'
    shutil.rmtree(vdir, ignore_errors=True)
    print(f'[{k}/{len(zip_names)}] {name}: downloading…', flush=True)
    if LOCAL_TEST_ZIPS: shutil.copy(os.path.join(LOCAL_TEST_ZIPS, name), zp)
    else: sh(f'wget -c -q -O "{zp}" "https://zenodo.org/records/4010759/files/{name}?download=1"')
    print('   unzipping…', flush=True); sh(f'unzip -q -o "{zp}" -d "{vdir}"'); os.remove(zp)
    print('   reading hand movements…', flush=True)
    L.extract_folder(vdir, f'{WORK}/landmarks', HAND, POSE, log=print)
    shutil.rmtree(vdir, ignore_errors=True)
    open(marker, 'w').write('ok')
print(f'Done in {(time.time()-t_all)/60:.0f} min.', len(os.listdir(f"{WORK}/landmarks")), 'videos ready.')

In [ ]:
#@title 4 · Prepare training data { display-mode: "form" }
AUGMENTED_COPIES_PER_VIDEO = 12  #@param {type:"slider", min:2, max:24, step:1}
from collections import Counter
items = L.load_landmarks(f'{WORK}/landmarks')
count = Counter(d['label'] for d in items)
labels = sorted(l for l, c in count.items() if c >= 4)          # need a few videos per sign
items = [d for d in items if d['label'] in labels]
# Optional: the official INCLUDE train/test split. Put train.csv and test.csv in the work folder to use it.
official = None
if os.path.exists(f'{WORK}/train.csv') and os.path.exists(f'{WORK}/test.csv'):
    import re
    names = lambda p: {m.lower() for m in re.findall(r'[\w\-. ]+\.(?:mov|mp4|avi|mkv)', open(p).read(), re.I)}
    official = ({os.path.basename(n) for n in names(f'{WORK}/train.csv')}, {os.path.basename(n) for n in names(f'{WORK}/test.csv')})
tr, te = L.split_items(items, test_frac=0.2, seed=0, official=official)
print(f'{len(labels)} signs, {len(items)} videos → {len(tr)} for training, {len(te)} held out for testing'
      + (' (official split)' if official else ' (random 80/20 split per sign)'))
t0 = time.time()
Xtr, Ytr, _, _ = L.build_arrays(items, labels, tr, n_aug=AUGMENTED_COPIES_PER_VIDEO, seed=1)
Xte, Yte, Xte_m, Yte_m = L.build_arrays(items, labels, te, n_aug=0, seed=2, mirror_test=True)
print(f'Training examples: {len(Xtr)} (incl. augmented copies), test: {len(Xte)}  [{time.time()-t0:.0f}s]')

In [ ]:
#@title 5 · Train the model { display-mode: "form" }
EPOCHS = 60  #@param {type:"slider", min:10, max:150, step:5}
t0 = time.time()
model = L.train(Xtr, Ytr, Xte, Yte, n_classes=len(labels), epochs=EPOCHS, log=print)
print(f'Training took {(time.time()-t0)/60:.1f} min')

In [ ]:
#@title 6 · How good is it? { display-mode: "form" }
r = L.evaluate(model, Xte, Yte, labels)
rm = L.evaluate(model, Xte_m, Yte_m) if len(Xte_m) else None
print(f'Held-out test videos: {r["n"]}')
print(f'  Correct first guess:      {r["top1"]*100:.1f}%')
print(f'  Correct within top 3:     {r["top3"]*100:.1f}%')
if rm: print(f'  Left-handed (mirrored):   {rm["top1"]*100:.1f}%')
print(f'  When the model is ≥50% sure ({r["confident_share"]*100:.0f}% of the time) it is right {r["confident_accuracy"]*100:.1f}% of the time')
print('\nHardest signs (accuracy):', ', '.join(f'{k} {v*100:.0f}%' for v, k in r['worst']))
print('Most common mix-ups:', '; '.join(r['confusions']) or 'none')
print('\nNote: INCLUDE does not say which student signed which video, so the same students can appear in training and')
print('test videos. Expect somewhat lower accuracy for new people and different cameras — test with real users.')

In [ ]:
#@title 7 · Export for the app and download { display-mode: "form" }
import zipfile, datetime
exp = f'{TMP}/export'; os.makedirs(exp, exist_ok=True)
meta = dict(dataset='INCLUDE (IIT Madras / AI4Bharat), CC BY 4.0 — https://zenodo.org/records/4010759',
            categories=cats if not LOCAL_TEST_ZIPS else ['local test'], train_videos=len(tr), test_videos=len(te),
            test_top1=round(r['top1'], 4), test_top3=round(r['top3'], 4), mirrored_top1=round(rm['top1'], 4) if rm else None,
            epochs=EPOCHS, created=datetime.datetime.now().isoformat(timespec='minutes'))
diff = L.export_onnx(model, f'{exp}/isl_model.onnx', labels, meta)
print('ONNX check (max difference vs PyTorch):', f'{diff:.2e}', '✓' if diff < 1e-3 else '✗ — please report this')
zpath = f'{WORK}/isl_model.zip'
with zipfile.ZipFile(zpath, 'w') as z:
    for f in ('isl_model.onnx', 'isl_model.json'): z.write(f'{exp}/{f}', f'model/{f}')
print('Saved', zpath, f'({os.path.getsize(zpath)/1e6:.1f} MB)')
if IN_COLAB:
    from google.colab import files; files.download(zpath)
print('Unzip it into your ISL_Avatar_App folder (it creates model/isl_model.onnx and model/isl_model.json), then restart the app.')